# Merging orders with products and customers

**Video 6.6** | ShopKart project

**Goal:** Join the cleaned orders with products and customers (like XLOOKUP and SQL LEFT JOIN), check no rows are lost or doubled, and save shopkart_clean.csv.

### How to open in Colab
1. Go to colab.research.google.com → **File → Upload notebook** → choose this `.ipynb` file.
2. Click the **folder icon** on the left sidebar (Files panel) and drag in the data files: `orders_clean.csv`, `products.csv`, `customers.csv`.
3. Run cells one by one with **Shift + Enter**.

> Files you upload to the Files panel disappear when the runtime resets. If you see `FileNotFoundError`, upload them again.

In [1]:
import pandas as pd
orders = pd.read_csv("orders_clean.csv", parse_dates=["order_date"])
products = pd.read_csv("products.csv")
customers = pd.read_csv("customers.csv")
print(orders.shape, products.shape, customers.shape)

(5000, 12) (60, 5) (800, 7)


## merge = SQL JOIN
| SQL (Module 4) | pandas |
|---|---|
| `FROM orders o LEFT JOIN products p ON o.product_id = p.product_id` | `orders.merge(products, on="product_id", how="left")` |

`how="left"` keeps **every order**, even if a product were missing. We only bring the columns we need.

In [2]:
# Before merging: are the keys unique in the lookup tables? (if not, rows would double)
print("Duplicate product_ids:", products["product_id"].duplicated().sum())
print("Duplicate customer_ids:", customers["customer_id"].duplicated().sum())

Duplicate product_ids: 0
Duplicate customer_ids: 0


In [3]:
df = orders.merge(products[["product_id", "product_name", "category", "cost_price"]],
                  on="product_id", how="left")
print(df.shape)   # must still be 5,000 rows

(5000, 15)


In [4]:
df = df.merge(customers[["customer_id", "gender", "age", "membership"]],
              on="customer_id", how="left")
print(df.shape)
assert len(df) == 5000, "Row count changed - check the join keys!"
print("Rows with no product match:", df["category"].isnull().sum())
print("Rows with no customer match:", df["membership"].isnull().sum())

(5000, 18)
Rows with no product match: 0
Rows with no customer match: 0


## Add the calculated columns (from 6.4) and save

In [5]:
df["revenue"] = (df["quantity"] * df["unit_price"] * (1 - df["discount"])).round(2)
df["profit"] = df["revenue"] - df["quantity"] * df["cost_price"]
df["order_month"] = df["order_date"].dt.month
print(df.shape)
print(list(df.columns))

(5000, 21)
['order_id', 'order_date', 'customer_id', 'product_id', 'quantity', 'unit_price', 'discount', 'city', 'payment_mode', 'delivery_days', 'rating', 'returned', 'product_name', 'category', 'cost_price', 'gender', 'age', 'membership', 'revenue', 'profit', 'order_month']


In [6]:
df.head()

,order_id,order_date,customer_id,product_id,quantity,unit_price,discount,city,payment_mode,delivery_days,...,returned,product_name,category,cost_price,gender,age,membership,revenue,profit,order_month
0,O10001,2025-01-01,C0593,P048,6,362,0.20,Delhi,COD,4,...,N,Wings of Fire,Books,240,Female,26,Prime,1737.6,297.6,1
1,O10002,2025-01-01,C0367,P049,2,300,0.05,Chennai,Debit Card,4,...,N,Face Wash,Beauty,150,Male,35,Free,570.0,270.0,1
2,O10003,2025-01-01,C0707,P020,1,2258,0.05,Delhi,Debit Card,2,...,N,Saree,Fashion,1280,Male,41,Prime,2145.1,865.1,1
3,O10004,2025-01-01,C0118,P026,2,2020,0.15,Lucknow,COD,4,...,Y,Pressure Cooker,Home,1260,Male,35,Free,3434.0,914.0,1
4,O10005,2025-01-01,C0238,P026,2,1937,0.10,Chennai,UPI,2,...,N,Pressure Cooker,Home,1260,Female,33,Free,3486.6,966.6,1


In [7]:
df.to_csv("shopkart_clean.csv", index=False)
print("Saved shopkart_clean.csv:", df.shape)

Saved shopkart_clean.csv: (5000, 21)


This file is the input for **Module 8 (charts) and Module 9 (machine learning)**. The Module 10 capstone pulls the same cleaned data straight from `shopkart.db` with SQL.

## What merge did NOT include: customers who never ordered
A left join from orders only keeps customers who appear in orders. To find customers with no orders, flip it.

In [8]:
check = customers.merge(orders[["customer_id"]].drop_duplicates(), on="customer_id",
                        how="left", indicator=True)
check["_merge"].value_counts()

_merge
both          758
left_only      42
right_only      0
Name: count, dtype: int64

`left_only` = customers who never ordered (a marketing opportunity). Same as the SQL LEFT JOIN ... WHERE o.order_id IS NULL from Module 4.

## Practice
Now that category is available: total revenue and total profit per category. Which category makes the most profit?

In [9]:
df.groupby("category")[["revenue", "profit"]].sum().round(0).sort_values("profit", ascending=False)

,revenue,profit
category,,
Fashion,3121346.0,1130546.0
Electronics,7331018.0,808468.0
Home,2655047.0,689997.0
Beauty,877640.0,375400.0
Books,719598.0,136338.0
